## *REM*

In [ ]:
from unidecode import unidecode

In [ ]:
rem_clasico = pd.read_csv(r"C:\Users\Hp\DENGUE\DATOS\REM_Clasico_Atlantico.csv", encoding='latin_1', low_memory=False)
rem_grave = pd.read_csv(r"C:\Users\Hp\DENGUE\DATOS\REM_Grave_Atlantico.csv", encoding='latin_1', low_memory=False)
shp = gpd.read_file(r"C:\Users\Hp\DENGUE\DATOS\atlantico_municipios.shp", encoding='utf_8')


In [ ]:
shp_atlantico = shp[shp["dpto_cnmbr"].str.upper().str.strip() == "ATLÁNTICO"]

In [ ]:
# === 4. Función para graficar panel ===
def graficar_panel_rem_atlantico(df, shp, evento=None):
    """
    Grafica un panel de mapas del REM de dengue en Atlántico por años,
    cada uno con su escala individual.
    """
    # Normalizar municipios en CSV
    df["municipio"] = df["municipio"].str.upper().str.strip().map(unidecode)

    # Normalizar municipios en shapefile
    shp_atlantico["mpio_cnmbr"] = shp_atlantico["mpio_cnmbr"].str.upper().str.strip().map(unidecode)

    anios = sorted(df["anio"].unique())

    ncols = 4
    nrows = (len(anios) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(20, 12))
    axes = axes.flatten()

    for i, anio in enumerate(anios):
        ax = axes[i]

        df_anio = df[df["anio"] == anio]

        # Merge: shapefile + datos del año
        gdf_anio = shp.merge(df_anio, left_on="mpio_cnmbr", right_on="municipio", how="left")
        gdf_anio["rem"] = gdf_anio["rem"].fillna(0)

        vmin, vmax = gdf_anio["rem"].min(), gdf_anio["rem"].max()

        gdf_anio.plot(
            column="rem",
            cmap="Blues",
            linewidth=0.5,
            edgecolor="0.8",
            vmin=vmin, vmax=vmax,
            ax=ax,
            legend=True,
            legend_kwds={"shrink": 0.6, "label": "REM"}
        )

        ax.set_title(f"Año {anio}", fontsize=12, fontweight="bold")
        ax.axis("off")

    # Ocultar subplots vacíos
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    titulo = f"REM del dengue {evento or 'general'} en Atlántico ({anios[0]}–{anios[-1]})"
    fig.suptitle(titulo, fontsize=16, fontweight="bold")

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()



In [ ]:
# === 5. Llamar función ===
graficar_panel_rem_atlantico(rem_clasico, shp_atlantico, evento="Clásico")

In [ ]:
# === 5. Llamar función ===
graficar_panel_rem_atlantico(rem_grave, shp_atlantico, evento="Grave")